In [17]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt

file_path = "/Users/kyle/stock-trading-analytics-/data/spy_5y.csv"
df_csv = pd.read_csv(file_path)

In [18]:
df_csv = df_csv.drop([0]).reset_index(drop=True)
df_csv = df_csv.rename(columns={'Price': 'Date'})
column_order = ['Date', 'Open', 'Close', 'High', 'Low', 'Volume']
df_reordered = df_csv[column_order].copy()
df_reordered.dropna(inplace=True) 
df = df_reordered
df = df_csv[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']].copy()

df['Open'] = pd.to_numeric(df_csv['Open'], errors='coerce')
df['Low'] = pd.to_numeric(df_csv['Low'], errors='coerce')
df['High'] = pd.to_numeric(df_csv['High'], errors='coerce')
df['Volume'] = pd.to_numeric(df_csv['Volume'], errors='coerce')
df['Close'] = pd.to_numeric(df_csv['Close'], errors='coerce')

In [19]:
df['Daily Return'] = df['Close'].pct_change()
df['Rolling Volatility'] = df['Daily Return'].rolling(window=21).std()

df['MA_10'] = df['Close'].rolling(window=10).mean()
df['MA_50'] = df['Close'].rolling(window=50).mean()
df['MA_200'] = df['Close'].rolling(window=200).mean()

df['H-L'] = df['High'] - df['Low']
df['H-PC'] = abs(df['High'] - df['Close'].shift(1))
df['L-PC'] = abs(df['Low'] - df['Close'].shift(1))
df['TR'] = df[['H-L', 'H-PC', 'L-PC']].max(axis=1)
df['ATR_14'] = df['TR'].rolling(window=14).mean()

In [20]:
# Momentum Features, Percent Change
df['Momentum_10'] = df['Close'].pct_change(periods=10)

# Momentum Features, MACD (12-day EMA minus 26-day EMA)
ema_12 = df['Close'].ewm(span=12, adjust=False).mean()
ema_26 = df['Close'].ewm(span=26, adjust=False).mean()
df['MACD'] = ema_12 - ema_26
df['Signal_Line'] = df['MACD'].ewm(span=9, adjust=False).mean()

# Momentum Features, RSI (14-day Relative Strength Index)
delta = df['Close'].diff()
gain = delta.where(delta > 0, 0)
loss = -delta.where(delta < 0, 0)
avg_gain = gain.rolling(window=14).mean()
avg_loss = loss.rolling(window=14).mean()
rs = avg_gain / avg_loss
df['RSI_14'] = 100 - (100 / (1 + rs))


In [21]:
# Saving engineered dataset 
df.dropna(inplace=True)
df.to_csv("/Users/kyle/stock-trading-analytics-/data/spy_features.csv")